# Deep Learning (Neural Networks): Theory, Math, and Implementation

Deep Learning is a specialized subset of Machine Learning inspired by the structure and function of the human brain. Instead of using a single mathematical formula to make predictions, it chains together layers of "artificial neurons" to learn incredibly complex, non-linear patterns. 

This is the technology driving the current AI revolution, powering everything from ChatGPT to self-driving cars.

---
### Setup & Imports
We will use a variety of datasets to demonstrate the different network architectures, including 2D coordinates for standard networks, time-series data for RNNs, and image data for CNNs and Autoencoders.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.datasets import make_moons, load_digits
from ipywidgets import interact, IntSlider, Dropdown, FloatLogSlider
import scipy.ndimage as ndimage
import scipy.datasets

plt.style.use('seaborn-v0_8-darkgrid')

# 1. Dataset for MLP (Non-linear Moons)
X_moons, y_moons = make_moons(n_samples=300, noise=0.15, random_state=42)

# 2. Dataset for Autoencoders (8x8 Image Digits)
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

# 3. Dataset for CNNs (Standard high-res sample image)
image_ascent = scipy.datasets.ascent().astype(float)

### 1. Multi-Layer Perceptron (MLP)

The Multi-Layer Perceptron is the classic, foundational Feed-Forward Neural Network. It consists of an **Input Layer**, one or more **Hidden Layers**, and an **Output Layer**. Every neuron in a layer is connected to every neuron in the next layer (Fully Connected).

**Mathematical Foundation:**
Neural networks learn through a two-step mathematical process:

**1. Forward Propagation (Making a guess):**
Data flows forward through the network. Each neuron calculates a weighted sum of its inputs, adds a bias, and passes the result through a non-linear **Activation Function** ($\sigma$) (like ReLU, Sigmoid, or Tanh) to decide if the neuron should "fire."
$$Z = W \cdot X + b$$
$$A = \sigma(Z)$$
*(Where $W$ are the weights, $b$ is the bias, and $A$ is the activation output).*

**2. Backpropagation (Learning from mistakes):**
The network compares its final output to the true target to calculate the **Loss** (Error). It then works backwards through the network using the **Chain Rule of Calculus** to figure out exactly how much each individual weight contributed to the error. It uses an optimizer (like Gradient Descent or Adam) to update the weights:
$$W_{new} = W_{old} - \eta \frac{\partial \text{Loss}}{\partial W}$$
*(Where $\eta$ is the learning rate).*

**Example Problem:**
* **Tabular Data:** Predicting if a patient has heart disease based on rows of medical history data (age, blood pressure, cholesterol).

Use the interactive visualization below to build your own MLP. Notice how a network with no hidden layers acts just like a rigid linear model, but adding layers allows it to warp and bend the decision boundary!

In [ ]:
@interact(
    hidden_neurons=IntSlider(min=1, max=100, step=5, value=10, description='Neurons/Layer'),
    hidden_layers=IntSlider(min=1, max=3, step=1, value=1, description='Hidden Layers'),
    activation=Dropdown(options=['relu', 'tanh', 'logistic'], value='relu', description='Activation')
)
def plot_mlp(hidden_neurons, hidden_layers, activation):
    layer_sizes = tuple([hidden_neurons] * hidden_layers)
    
    mlp = MLPClassifier(hidden_layer_sizes=layer_sizes, activation=activation, max_iter=1000, random_state=42)
    mlp.fit(X_moons, y_moons)
    
    # Visualization Grid
    x_min, x_max = X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5
    y_min, y_max = X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
    
    Z = mlp.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(8, 5))
    plt.contourf(xx, yy, Z, levels=10, cmap='RdBu', alpha=0.6)
    plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='RdBu', edgecolors='k')
    
    plt.title(f"MLP Decision Boundary\nArchitecture: {hidden_layers} Layers, {hidden_neurons} Neurons each ({activation.upper()})", fontsize=14)
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

interactive(children=(IntSlider(value=10, description='Neurons/Layer', min=1, step=5), IntSlider(value=1, desc…

### 2. Convolutional Neural Networks (CNN)

If you feed a high-resolution image into a standard MLP, it treats every single pixel as an isolated feature, completely destroying the spatial structure of the picture (a cat's ear is made of pixels that are right next to each other!).

CNNs solve this using **Filters (Kernels)**. Instead of looking at the whole image at once, a CNN slides a small grid (e.g., 3x3 pixels) across the image. This process is called **Convolution**. 
Because it scans the image, a CNN achieves **Translation Invariance**: it can recognize a cat whether it is in the top-left corner or the bottom-right corner!

**Mathematical Foundation:**
A convolution mathematically multiplies a matrix of pixels by a matrix of weights (the filter) and sums the result to create a new **Feature Map**.
$$S(i, j) = (I * K)(i, j) = \sum_m \sum_n I(i-m, j-n) K(m, n)$$
*(Where $I$ is the input image and $K$ is the kernel).*

In deep learning, we do not manually design these filters. The CNN *learns* the exact numbers in the filter matrix to extract whatever features (edges, textures, shapes) are most useful for the task!

**Example Problem:**
* **Computer Vision:** Image classification (Cat vs. Dog), object detection (YOLO algorithm for self-driving cars), and medical image analysis (identifying tumors in MRI scans).

The visualization below demonstrates exactly how math matrices act as visual filters!

In [ ]:
@interact(filter_type=Dropdown(options=['Identity', 'Edge Detection (Outline)', 'Sharpen', 'Box Blur'], value='Edge Detection (Outline)', description='Filter Type'))
def plot_convolution(filter_type):
    kernels = {
        'Identity': np.array([[0, 0, 0], 
                              [0, 1, 0], 
                              [0, 0, 0]]),
        
        'Edge Detection (Outline)': np.array([[-1, -1, -1], 
                                              [-1,  8, -1], 
                                              [-1, -1, -1]]),
        
        'Sharpen': np.array([[ 0, -1,  0], 
                             [-1,  5, -1], 
                             [ 0, -1,  0]]),
        
        'Box Blur': np.array([[1, 1, 1], 
                              [1, 1, 1], 
                              [1, 1, 1]]) / 9.0
    }
    
    kernel = kernels[filter_type]
    
    convolved_image = ndimage.convolve(image_ascent, kernel)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
    
    ax1.imshow(image_ascent, cmap='gray')
    ax1.set_title("Original Input Image", fontsize=14)
    ax1.axis('off')
    
    ax2.imshow(convolved_image, cmap='gray')
    ax2.set_title(f"Feature Map (After {filter_type} Filter)", fontsize=14)
    ax2.axis('off')
    
    matrix_str = str(np.round(kernel, 2)).replace('[', '').replace(']', '')
    fig.text(0.5, 0.05, f"Mathematical Kernel Used:\n{matrix_str}", ha='center', fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

interactive(children=(Dropdown(description='Filter Type', index=1, options=('Identity', 'Edge Detection (Outli…

### 3. Recurrent Neural Networks (RNN / LSTM / GRU)

Standard networks assume that all inputs are completely independent. But what if you are trying to predict the next word in a sentence, or the stock market price for tomorrow? The *sequence* and *history* matter.

RNNs solve this by adding a **"Memory Loop."** When an RNN processes data at time step $t$, it looks at the current input *and* its own output from time step $t-1$. 

**The Vanishing Gradient Problem & LSTMs:**
Basic RNNs have "short-term memory loss." During backpropagation, the gradients multiply rapidly. If the numbers are small, they vanish to 0, and the network forgets early data in a long sequence. 
**Long Short-Term Memory (LSTM)** networks fix this by introducing a "Conveyor Belt" (Cell State) and mathematical **Gates** (Forget, Input, Output) that learn exactly what information to keep and what to throw away.

**Mathematical Foundation (Basic RNN):**
$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b)$$
*(The hidden state $h$ at time $t$ is a combination of the previous hidden state $h_{t-1}$ and the current input $x_t$).*

**Example Problem:**
* **Time-Series & Sequences:** Speech recognition (Siri/Alexa), stock market prediction, and simple text generation.

In [ ]:
time = np.linspace(0, 20, 100)
wave = np.sin(time) + np.sin(2 * time)

@interact(look_back=IntSlider(min=1, max=20, step=1, value=5, description='Memory Steps'))
def plot_rnn_memory(look_back):
    plt.figure(figsize=(12, 4))
    
    plt.plot(time, wave, color='lightgray', linestyle='--', label='Full Future Sequence')
    
    current_idx = 40
    
    plt.plot(time[current_idx-look_back : current_idx], 
             wave[current_idx-look_back : current_idx], 
             color='blue', linewidth=3, label=f'RNN Memory (Past {look_back} steps)')

    plt.scatter(time[current_idx], wave[current_idx], color='red', s=100, zorder=5, label='Target Prediction')
    
    plt.title(f"RNN Sequence Processing (Looking back {look_back} steps to predict the future)", fontsize=14)
    plt.legend()
    plt.show()

interactive(children=(IntSlider(value=5, description='Memory Steps', max=20, min=1), Output()), _dom_classes=(…

### 4. Transformers (The Attention Mechanism)

While RNNs process text sequentially (word-by-word), **Transformers** process entire sequences simultaneously. They rely entirely on a mechanism called **Self-Attention**. 

Self-Attention allows the model to look at a single word and instantly calculate how important *every other word in the sentence* is to understanding its context. Because it processes everything in parallel, it can be trained on massive datasets (like the entire internet), leading to the creation of Large Language Models (LLMs).

**Mathematical Foundation (Scaled Dot-Product Attention):**
Every word in a sentence is converted into three vectors: a Query ($Q$), a Key ($K$), and a Value ($V$).
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$
1. **$Q \cdot K^T$:** The model calculates the dot-product between the Query of the current word and the Keys of all other words. This acts as a "Compatibility Score" (how closely related the words are).
2. **$\text{softmax}$:** This normalizes the scores so they add up to 100%.
3. **$\cdot V$:** The model multiplies these percentages by the Value vectors to assemble the final, context-aware meaning of the word!

**Example Problem:**
* **NLP & GenAI:** Google Translate, Large Language Models (GPT-4, BERT, Gemini), and even biology (AlphaFold predicting 3D protein structures).

In the interactive visualization below, look at the matrix to see how the Transformer "Attends" to the context of the sentence to figure out what the word "it" refers to!

In [ ]:
sentence = ["The", "cat", "sat", "on", "the", "mat", "because", "it", "was", "tired"]

@interact(focus_word=Dropdown(options=sentence, value='it', description='Focus Word'))
def plot_attention_matrix(focus_word):
    attention_scores = np.random.rand(len(sentence)) * 0.1 
    
    focus_idx = sentence.index(focus_word)
    attention_scores[focus_idx] = 1.0 
    
    if focus_word == "it":
        attention_scores[sentence.index("cat")] = 0.85
        attention_scores[sentence.index("tired")] = 0.60
    elif focus_word == "tired":
        attention_scores[sentence.index("cat")] = 0.70
        attention_scores[sentence.index("it")] = 0.60
    elif focus_word == "cat":
        attention_scores[sentence.index("sat")] = 0.50
        attention_scores[sentence.index("tired")] = 0.40
        
    attention_scores = attention_scores / np.sum(attention_scores)
    
    fig, ax = plt.subplots(figsize=(10, 2))
    cax = ax.matshow([attention_scores], cmap='Reds')
    
    ax.set_xticks(range(len(sentence)))
    ax.set_xticklabels(sentence, fontsize=12)
    ax.set_yticks([])
    
    plt.title(f"Self-Attention Scores: What does the model look at to understand '{focus_word}'?", pad=20, fontsize=14)
    plt.colorbar(cax, orientation='horizontal', fraction=0.2, pad=0.2, label='Attention Weight (Importance)')
    plt.show()

interactive(children=(Dropdown(description='Focus Word', index=7, options=('The', 'cat', 'sat', 'on', 'the', '…

### 5. Autoencoders

An Autoencoder is an unsupervised neural network designed to compress data and then perfectly reconstruct it. It looks like an hourglass:
1. **The Encoder:** Squeezes the high-dimensional input data through a tiny hidden layer called the **Bottleneck** (Latent Space).
2. **The Decoder:** Takes the compressed bottleneck data and attempts to rebuild the original input from scratch.

Because the network is forced to push the data through a tiny pipe, it cannot just memorize the input. It is forced to learn the absolute most essential, critical features of the dataset (Dimensionality Reduction).

**Mathematical Foundation:**
The network is trained to minimize the **Reconstruction Loss** (usually Mean Squared Error) between the original input $x$ and the rebuilt output $\hat{x}$:
$$L(x, \hat{x}) = || x - \hat{x} ||^2$$

**Example Problem:**
* **Anomaly / Fraud Detection:** You train an Autoencoder *only* on normal, legal credit card transactions. When a fraudulent transaction occurs, it looks mathematically different. The Autoencoder will fail to compress and rebuild it properly, resulting in a massive "Reconstruction Error," instantly flagging the transaction as an anomaly!

The visualization below trains an actual neural network to compress 64-pixel images of handwritten digits through a tiny bottleneck, and then attempts to redraw them. Watch what happens when the bottleneck is too small!

In [9]:
@interact(bottleneck_size=IntSlider(min=2, max=128, step=2, value=16, description='Bottleneck Size'))
def plot_autoencoder(bottleneck_size):
    autoencoder = MLPRegressor(hidden_layer_sizes=(bottleneck_size,), 
                               activation='relu', solver='adam', max_iter=300, random_state=42)

    autoencoder.fit(X_digits, X_digits)
    reconstructed_digits = autoencoder.predict(X_digits)

    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    
    for i in range(5):
        axes[0, i].imshow(X_digits[i].reshape(8, 8), cmap='gray')
        axes[0, i].axis('off')
        if i == 2: axes[0, i].set_title("Original Images (64 Pixels)\n", fontsize=14, fontweight='bold')
            
        axes[1, i].imshow(reconstructed_digits[i].reshape(8, 8), cmap='gray')
        axes[1, i].axis('off')
        if i == 2: axes[1, i].set_title(f"Reconstructed (Compressed to {bottleneck_size} numbers)\n", fontsize=14, fontweight='bold')
            
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=16, description='Bottleneck Size', max=128, min=2, step=2), Output()), _…